# Graph-Q-SAT and GAT-Q-SAT
The experiments of the note in [`report/`](https://github.com/dmeoli/GQSAT/tree/main/report): Graph-Q-SAT
(Kurin et al., 2020) and GAT-Q-SAT, which adds two edge-featured graph attention layers to its core block,
trained by Deep Q-Learning to reduce the number of decisions of MiniSat, together with two controls that
separate the effect of the attention from that of the added parameters.

*Runtime → Change runtime type → GPU*, then *Run all*. The checkpoints, the logs and the figures are kept on
Drive (`MyDrive/GQSAT_ckpts`, or the folder set in `CKPT_DIR` below), and whatever is already there is
skipped: a finished training exits at once, an interrupted one resumes from its last checkpoint, a complete
evaluation log is not computed again, and a run that is still training is not evaluated. After a disconnect,
*Run all* again continues from where it stopped.

## 1. Setup

In [ ]:
!nvidia-smi -L || echo 'no GPU: Runtime > Change runtime type > GPU'
import os
try:
    from google.colab import drive
    drive.mount('/content/gdrive', force_remount=True)
    CKPT_DIR = '/content/gdrive/MyDrive/GQSAT_ckpts'
except Exception as e:
    print('Drive not mounted (%s): the checkpoints stay on this machine' % e)
    CKPT_DIR = '/content/GQSAT_ckpts'
os.makedirs(CKPT_DIR, exist_ok=True); print('checkpoints ->', CKPT_DIR)

if not os.path.isdir('/content/GQSAT/.git'):
    !git clone --recurse-submodules https://github.com/dmeoli/GQSAT /content/GQSAT
%cd /content/GQSAT
!git pull origin main 2>&1 | tail -1
!git submodule update --init --recursive 2>&1 | tail -2

The current releases of PyTorch and PyTorch Geometric, and the SWIG wrapper of MiniSat built against the
installed `numpy`.

In [ ]:
!pip -q install -r requirements.txt
!sudo apt-get -qq install -y swig zlib1g-dev >/dev/null
import sys, numpy
PYV = f'python{sys.version_info.major}.{sys.version_info.minor}'
!sudo apt-get -qq install -y {PYV}-dev >/dev/null
%cd /content/GQSAT/minisat
!make python-wrap PYTHON={PYV} NUMPY_INC="{numpy.get_include()}" 2>&1 | tail -2
%cd /content/GQSAT
import minisat
print('environment built:', os.path.exists('minisat/minisat/gym/_GymSolver.so'))

## 2. Instances and MiniSat baselines
The instances are in `data/`, from SATLIB: graph coloring (`flat*`), uniform random 3-SAT (`uf*`, `uuf*`), with
the training, validation and test splits of the families used in training, and the structured families of the
transfer panel (`aim`, `planning`, `satlib/*`). The MRIR divides the iterations of MiniSat by those of the
model, and MiniSat is run with and without restarts on every instance; the iterations are stored with each
family (`METADATA`). For the SATLIB families they are computed here, on the CPU with a limit of 10 minutes per
instance (an instance not solved within it is listed in `EXCLUDED` and left out), and mirrored on Drive, so
that a family already done is skipped.

In [ ]:
%cd /content/GQSAT
!python make_metadata.py data/satlib/* --timeout 600 --mirror {CKPT_DIR}/metadata

## 3. Evaluation of the runs already trained
Every evaluation writes one line per instance, with the time, the iterations of the model and those of both
MiniSat runs. The phases: `seeds` (the coloring-trained runs of 2026, the controls included), the transfer
panel, `aa` (two evaluations repeated, for the noise of the running times) and `colouring` (the cap sweep of
the runs of 2021, which are in `runs/`). This cell comes before the training, so that a session first spends
its time on what is already possible; the runs not trained yet are skipped, and evaluated in section 5.

In [ ]:
%cd /content/GQSAT
import glob
REEVAL_DIR = f'{CKPT_DIR}/reeval'; os.makedirs(REEVAL_DIR, exist_ok=True)
# MIN_FREE_MB=0 disables the memory guard of a shared machine, DEVICE_FLAG= runs
# the network on the GPU, CKPT_ROOT is where the runs of 2026 are
ENV = f'OUT_ROOT={REEVAL_DIR} CKPT_ROOT={CKPT_DIR} DEVICE_FLAG= MIN_FREE_MB=0 THREADS=4 PY=python'
def evaluate_all():
    """Every evaluation of the note; complete logs and runs still training are skipped."""
    get_ipython().system(f'{ENV} bash reeval.sh seeds')
    get_ipython().system(f'OUT_ROOT={CKPT_DIR}/transfer_logs CKPT_ROOT={CKPT_DIR} '
                         f'PAPER_IMG_DIR={CKPT_DIR}/transfer DEVICE_FLAG= python transfer_study.py')
    get_ipython().system(f'{ENV} bash reeval.sh aa')
    get_ipython().system(f'{ENV} bash reeval.sh colouring')
    for d in sorted(glob.glob(f'{REEVAL_DIR}/*/')):
        print(os.path.basename(d.rstrip('/')), len(glob.glob(d + '*.tsv')), 'logs')
evaluate_all()

## 4. Training
`train.sh` has the configuration of Kurin et al. (Adam at $2\cdot 10^{-5}$, 50000 updates, a cap of 500
decisions per episode, a penalty of 0.1 per step, the gradient clipped at 0.1, and so on); each run has its own
folder on Drive, resumes from its `status.yaml`, and exits at once when it is finished. The runs of the note,
in order of importance: Graph-Q-SAT and GAT-Q-SAT (not seeded), one seed of each control (`graphwide`, hidden
layers of 104 units, the capacity of GAT-Q-SAT with no attention; `attnagg`, attention in place of the sum, at
the capacity of Graph-Q-SAT), a third seed of the two models, and a second seed of each control. A run takes
about 11 to 12 hours on a T4.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir {CKPT_DIR}

In [ ]:
%cd /content/GQSAT
TRAIN = 'data/graph-coloring/train/flat50-115'
VAL   = 'data/graph-coloring/val/flat50-115'
for variant in ['graphqsat', 'gatqsat']:
    print(f'\n##### training {variant} #####')
    !bash train.sh {variant} {TRAIN} {VAL} {CKPT_DIR}/gqsat_{variant}
for variant, seed in [('graphwide', 1), ('attnagg', 1), ('graphqsat', 3), ('gatqsat', 3),
                      ('graphwide', 2), ('attnagg', 2)]:
    print(f'\n##### training {variant}, seed {seed} #####')
    !SEED={seed} bash train.sh {variant} {TRAIN} {VAL} {CKPT_DIR}/gqsat_{variant}_s{seed}

## 5. Evaluation of the runs just trained
The evaluations of section 3 again: what is done is skipped, and the runs finished in section 4 are evaluated.

In [ ]:
evaluate_all()

## 6. Tables and figures
The tables of the note (MiniSat with restarts in the denominator on graph coloring, without on uniform random
3-SAT), the same logs against MiniSat without restarts everywhere, and the figures, drawn from the same logs
and written to Drive (`figures/`, the transfer ones in `transfer/`).

In [ ]:
%cd /content/GQSAT
!OUT_ROOT={REEVAL_DIR} python paper_numbers.py
!OUT_ROOT={REEVAL_DIR} python paper_numbers.py --baseline no_restarts
!OUT_ROOT={REEVAL_DIR} python make_plots.py --out-dir {CKPT_DIR}/figures
!OUT_ROOT={REEVAL_DIR} PAPER_IMG_DIR={CKPT_DIR}/figures python paper_analysis.py
from IPython.display import Image, display
for f in ['graphqsat', 'gatqsat', 'thesis', 'generalization', 'graphqsat_random', 'gatqsat_random']:
    display(Image(f'{CKPT_DIR}/figures/{f}.png', width=600))
for f in ['transfer', 'transfer_sw']:
    if os.path.exists(f'{CKPT_DIR}/transfer/{f}.png'):
        display(Image(f'{CKPT_DIR}/transfer/{f}.png', width=600))